In [1]:
import sqlite3, pandas as pd, random
from datetime import date, timedelta

random.seed(42)
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE sales (id INT, product TEXT, amount REAL, sale_date TEXT)')
for i in range(100):
    conn.execute('INSERT INTO sales VALUES (?,?,?,?)',
        (i, random.choice(['A','B','C']), round(random.uniform(10, 200), 2),
         str(date(2024,1,1) + timedelta(days=i*3))))
conn.commit()
print("Sales table ready: 100 rows")

Sales table ready: 100 rows


In [2]:
q1 = pd.read_sql_query("""
    SELECT
        id,
        product,
        amount,
        sale_date,
        SUM(amount) OVER (ORDER BY sale_date) AS running_total,
        RANK() OVER (ORDER BY amount DESC) AS revenue_rank
    FROM sales
""", conn)

q1.head(10)


,id,product,amount,sale_date,running_total,revenue_rank
0,0,C,31.15,2024-01-01,31.15,88
1,1,C,62.26,2024-01-04,93.41,69
2,2,A,36.51,2024-01-07,129.92,84
3,3,A,138.57,2024-01-10,268.49,39
4,4,C,26.52,2024-01-13,295.01,91
5,5,B,16.04,2024-01-16,311.05,99
6,6,A,51.54,2024-01-19,362.59,76
7,7,C,124.38,2024-01-22,486.97,45
8,8,C,47.78,2024-01-25,534.75,79
9,9,C,143.25,2024-01-28,678.00,33


In [3]:
top_per_product = pd.read_sql_query("""
    WITH ranked_sales AS (
        SELECT
            id,
            product,
            amount,
            sale_date,
            ROW_NUMBER() OVER (PARTITION BY product ORDER BY amount DESC) AS rn
        FROM sales
    )
    SELECT id, product, amount, sale_date
    FROM ranked_sales
    WHERE rn = 1
""", conn)

top_per_product


,id,product,amount,sale_date
0,15,A,191.87,2024-02-15
1,27,B,197.19,2024-03-22
2,33,C,187.96,2024-04-09


In [4]:
assert len(q1) > 0, "Q1 empty"
assert q1['running_total'].is_monotonic_increasing, "Running total should increase"
assert len(top_per_product) == 3, "Should have 1 top sale per product"
print("All checks passed!")


All checks passed!
